# Phase 1 — Lesson 16: Sampling Methods
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 16: Sampling Methods
**Type:** Build | **Languages:** Python | **Time:** ~120 minutes
**Prerequisites:** Phase 1, Lessons 06-07 (Probability, Bayes' Theorem)



# What You'll Learn

---

## Learning Objectives

- Implement inverse CDF, rejection, and importance sampling from scratch using only uniform random numbers
- Build temperature, top-k, and top-p (nucleus) sampling for language model token generation
- Explain the reparameterization trick and why it enables backpropagation through sampling in VAEs
- Run Metropolis-Hastings MCMC to sample from an unnormalized target distribution


# The Problem

---

A language model produces a vector of 50,000 logits — one for every token. Now it has to pick one.

If it always picks the highest-probability token, every response is identical. If it picks uniformly at random, the output is gibberish.

Sampling is how AI explores the space of possibilities — token generation, RL actions, diffusion models, MCMC.


# Uniform Random Sampling

---

Every sampling method starts here. A uniform random number generator produces values in [0, 1):

```
U ~ Uniform(0, 1)
P(a ≤ U ≤ b) = b - a
```


In [ ]:
import randomimport numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)random.seed(42)samples = [random.random() for _ in range(10000)]print(f"Mean: {np.mean(samples):.4f} (expected 0.5)")print(f"Std:  {np.std(samples):.4f} (expected ~0.289)")plt.hist(samples, bins=20, color='#2c3e50', alpha=0.7, edgecolor='white')plt.title('Uniform(0, 1) Samples', fontsize=12, fontweight='bold')plt.xlabel('Value'); plt.ylabel('Count'); plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Inverse CDF Sampling

---

If U ~ Uniform(0, 1), then X = F⁻¹(U) follows the target distribution.

**Exponential distribution:** F(x) = 1 − e^(−λx), so F⁻¹(u) = −ln(1−u)/λ = −ln(u)/λ.


In [ ]:
import mathimport randomimport numpy as npimport matplotlib.pyplot as pltdef sample_exponential_inverse_cdf(lam, n=10000):    return [-math.log(random.random()) / lam for _ in range(n)]samples = sample_exponential_inverse_cdf(lam=2.0)print(f"Mean: {np.mean(samples):.4f} (expected 1/2 = 0.5)")plt.hist(samples, bins=50, density=True, alpha=0.6, color='#2c3e50', label='Samples')xs = np.linspace(0, 5, 200)plt.plot(xs, 2*np.exp(-2*xs), 'r-', lw=2, label='True Exp(2)')plt.title('Inverse CDF Sampling: Exponential', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Rejection Sampling

---

Sample from a complex distribution P using a simpler proposal Q:

1. Draw x from Q
2. Draw u ~ Uniform(0, 1)
3. Accept x if u < P(x) / (M · Q(x))
4. Otherwise reject and repeat


In [ ]:
import randomimport mathimport numpy as npimport matplotlib.pyplot as pltrandom.seed(42)np.random.seed(42)def target_pdf(x):    return 0.5 * math.exp(-(x - 2) ** 2 / 2) + 0.5 * math.exp(-(x + 2) ** 2 / 2)def rejection_sampling(n_samples=5000):    samples = []    attempts = 0    while len(samples) < n_samples:        attempts += 1        x = random.uniform(-5, 5)        u = random.random()        if u < target_pdf(x) / (5 * 0.1):  # M=5, proposal pdf=0.1            samples.append(x)    return samples, attempts / n_samplessamples, inv_rate = rejection_sampling()print(f"Acceptance rate: {1/inv_rate:.3f}")plt.hist(samples, bins=50, density=True, alpha=0.6, color='#2c3e50', label='Samples')xs = np.linspace(-5, 5, 200)plt.plot(xs, [target_pdf(x) for x in xs], 'r-', lw=2, label='Target PDF')plt.title('Rejection Sampling: Bimodal Distribution', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Importance Sampling

---

Estimate E_P[f(X)] using samples from Q, weighted by P/Q:

```
E_P[f(X)] ≈ (1/N) Σ f(xᵢ) · P(xᵢ) / Q(xᵢ)
```


In [ ]:
import randomimport mathimport numpy as nprandom.seed(42)def target_pdf(x):    return math.exp(-x ** 2 / 2) / math.sqrt(2 * math.pi)def proposal_pdf(x):    return 1 / 6 if -3 <= x <= 3 else 0def proposal_sample():    return random.uniform(-3, 3)def f(x):    return x ** 2def importance_sampling(n=100000):    total = 0.0    wtotal = 0.0    for _ in range(n):        x = proposal_sample()        w = target_pdf(x) / proposal_pdf(x)        total += f(x) * w        wtotal += w    return total / wtotalestimate = importance_sampling()print(f"E[X²] estimate: {estimate:.4f} (true value = 1.0)")

# Monte Carlo Estimation of π

---

Approximate integrals by averaging random samples. Error is O(1/√N) regardless of dimension.

Sample (x, y) uniformly from [−1, 1] × [−1, 1]. Count points in unit circle: π ≈ 4 × (inside / total).


In [ ]:
import randomimport mathimport numpy as npimport matplotlib.pyplot as pltrandom.seed(42)def monte_carlo_pi(n):    inside = 0    xs_in, ys_in = [], []    xs_out, ys_out = [], []    for _ in range(n):        x = random.uniform(-1, 1)        y = random.uniform(-1, 1)        if x*x + y*y <= 1:            inside += 1            xs_in.append(x); ys_in.append(y)        else:            xs_out.append(x); ys_out.append(y)    return 4 * inside / n, xs_in, ys_in, xs_out, ys_outpi_est, xs_in, ys_in, xs_out, ys_out = monte_carlo_pi(10000)print(f"π estimate: {pi_est:.4f} (true π = {math.pi:.4f})")plt.figure(figsize=(6, 6))plt.scatter(xs_in, ys_in, c='green', s=2, alpha=0.5, label='Inside')plt.scatter(xs_out, ys_out, c='red', s=2, alpha=0.5, label='Outside')circle = plt.Circle((0, 0), 1, fill=False, color='black', lw=2)plt.gca().add_patch(circle)plt.xlim(-1.1, 1.1); plt.ylim(-1.1, 1.1)plt.setp(plt.gca(), aspect='equal')plt.title(f'Monte Carlo π ≈ {pi_est:.4f}', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Markov Chain Monte Carlo: Metropolis-Hastings

---

Construct a Markov chain whose stationary distribution is the target p(x):

1. Propose x′ from proposal Q(x′\|x)
2. Accept with probability min(1, p(x′)/p(x)) (for symmetric Q)
3. Repeat — chain converges to p(x)


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef metropolis_hastings(target_log_prob, x0, n_samples=10000, proposal_std=0.5, seed=42):    rng = np.random.RandomState(seed)    x = x0    samples = [x]    for _ in range(n_samples):        x_prop = x + rng.randn() * proposal_std        log_ratio = target_log_prob(x_prop) - target_log_prob(x)        if math.log(rng.rand()) < log_ratio:            x = x_prop        samples.append(x)    return np.array(samples)def target_log_prob(x):    # Bimodal: mixture of two Gaussians    return math.log(0.5 * math.exp(-(x - 2)**2 / 2) + 0.5 * math.exp(-(x + 2)**2 / 2))samples = metropolis_hastings(target_log_prob, x0=0.0)print(f"Mean: {np.mean(samples):.4f}")print(f"Std:  {np.std(samples):.4f}")plt.figure(figsize=(7, 4))plt.hist(samples, bins=50, density=True, alpha=0.6, color='#2c3e50', label='MH samples')xs = np.linspace(-5, 5, 200)target = [0.5 * math.exp(-(x - 2)**2 / 2) + 0.5 * math.exp(-(x + 2)**2 / 2) for x in xs]plt.plot(xs, target, 'r-', lw=2, label='Target PDF')plt.title('Metropolis-Hastings: Bimodal Target', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Temperature Sampling (Used in LLMs)

---

Language models output logits zᵢ. Temperature rescales before softmax:

```
pᵢ = exp(zᵢ / T) / sum(exp(zⱼ / T))

T < 1: sharper (more confident).  T > 1: flatter (more diverse).  T → 0: greedy.
```


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef temperature_sample(logits, temperature):    scaled = np.array(logits) / temperature    scaled = scaled - scaled.max()    exps = np.exp(scaled)    return exps / exps.sum()logits = [2.0, 1.0, 0.0, -1.0]for t in [0.1, 0.5, 1.0, 2.0]:    p = temperature_sample(logits, t)    print(f"T={t}: {np.round(p, 4)}")fig, ax = plt.subplots(figsize=(7, 4))tokens = [0, 1, 2, 3]for t, color in [(0.1, '#e74c3c'), (0.5, '#f39c12'), (1.0, '#27ae60'), (2.0, '#3498db')]:    p = temperature_sample(logits, t)    ax.bar([i + 0.1 * (tokens.index(t) if False else 0) for i in tokens], p, alpha=0.6, label=f'T={t}', width=0.15)ax.set_xticks(tokens)ax.set_xticklabels([f'Token {i}' for i in tokens])ax.set_ylabel('Probability')ax.set_title('Temperature Sampling', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3, axis='y')plt.tight_layout(); plt.show()

# Top-k and Top-p (Nucleus) Sampling

---

**Top-k:** keep only the k highest-probability tokens, renormalize.
**Top-p:** keep the smallest set whose cumulative probability exceeds p.


In [ ]:
import numpy as npdef top_k_sample(logits, k):    probs = np.exp(logits - logits.max())    probs = probs / probs.sum()    top_idx = np.argsort(probs)[::-1][:k]    top_probs = probs[top_idx]    top_probs = top_probs / top_probs.sum()    return np.random.choice(top_idx, p=top_probs)def top_p_sample(logits, p):    probs = np.exp(logits - logits.max())    probs = probs / probs.sum()    sorted_idx = np.argsort(probs)[::-1]    sorted_probs = probs[sorted_idx]    cumulative = np.cumsum(sorted_probs)    cutoff = np.searchsorted(cumulative, p) + 1    top_idx = sorted_idx[:cutoff]    top_probs = probs[top_idx]    top_probs = top_probs / top_probs.sum()    return np.random.choice(top_idx, p=top_probs)logits = np.array([2.0, 1.0, 0.5, 0.1, -1.0])np.random.seed(42)print(f"Top-k (k=2):  {top_k_sample(logits, 2)}")print(f"Top-p (p=0.9): {top_p_sample(logits, 0.9)}")

# Reparameterization Trick (VAEs)

---

Cannot backpropagate through sampling. The trick:

```
z = μ + σ · ε,   where ε ~ N(0, 1)
```

Now z is a deterministic, differentiable function of μ and σ.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef reparameterize(mu, sigma, epsilon):    return mu + sigma * epsilonmu, sigma = 0.0, 1.0epsilons = np.random.randn(10000)samples = reparameterize(mu, sigma, epsilons)print(f"Mean: {np.mean(samples):.4f} (expected 0)")print(f"Std:  {np.std(samples):.4f} (expected 1)")plt.hist(samples, bins=50, density=True, alpha=0.6, color='#2c3e50', label='Reparameterized samples')xs = np.linspace(-4, 4, 200)plt.plot(xs, np.exp(-xs**2/2) / np.sqrt(2*np.pi), 'r-', lw=2, label='True N(0,1)')plt.title('Reparameterization Trick: z = μ + σ·ε', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** Why does temperature T → 0 give greedy decoding?
> **Explanation:** exp(zᵢ/T) amplifies differences — the highest logit dominates.

**Q2:** Why does top-p adapt better than top-k?
> **Explanation:** When the model is confident, nucleus sampling keeps few tokens; when uncertain, it keeps many.

**Q3:** Why is the reparameterization trick needed for VAEs?
> **Explanation:** You cannot backpropagate through a random sample; z = μ + σ·ε moves randomness to a parameter-free ε.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Implement inverse CDF sampling for the Cauchy distribution.

**Exercise 2:** Build a text generator using temperature, top-k, and top-p sampling; compare diversity.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Inverse CDF | F⁻¹(U) | Convert uniform sample to any distribution with known CDF |
| Rejection sampling | Propose and accept | Accept x with probability P(x)/(M·Q(x)) |
| Importance sampling | Reweight samples | E_P[f] ≈ Σ f(xᵢ)·P(xᵢ)/Q(xᵢ) |
| MCMC | Markov chain to target | Chain whose stationary distribution is p(x) |
| Metropolis-Hastings | Propose and accept/reject | MCMC using acceptance ratios |
| Temperature | Confidence knob | Divide logits by T before softmax |
| Reparameterization trick | z = μ + σ·ε | Make sampling differentiable for VAE training |


# Further Reading

---

- [Holtzman et al. (2020): Nucleus Sampling](https://arxiv.org/abs/1904.09751)
- [Kingma & Welling (2014): VAEs](https://arxiv.org/abs/1312.6114)


# Next Steps

---

When you're ready, continue to Lesson 17: Linear Systems.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
